# Análisis y Calidad de Datos - Licores

Este notebook contiene el análisis exploratorio, la **calidad de datos** y la **preparación de tablas** (DuckDB, CTEs, agregaciones, particionado, caches) que alimentan al notebook `forecasting-licores.ipynb`.

En `forecasting-licores.ipynb` solo se leen los Parquet generados acá y se entrenan/evalúan los modelos de predicción.

## Configuración inicial

In [26]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [27]:
# Carga de datos - Iowa Liquor Sales
DATA_PATH = "Iowa_Liquor_Sales.csv"
SAMPLE_SIZE = 200_000

df_raw = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE)
print(f"Dataset: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
df_raw.head(3)

Dataset: 200,000 filas × 24 columnas


,Invoice/Item Number,Date,Store Number,Store Name,Address,City,Zip Code,Store Location,County Number,County,...,Item Number,Item Description,Pack,Bottle Volume (ml),State Bottle Cost,State Bottle Retail,Bottles Sold,Sale (Dollars),Volume Sold (Liters),Volume Sold (Gallons)
0,S29198800001,11/20/2015,2191,Keokuk Spirits,1013 MAIN,KEOKUK,52632,"1013 MAIN\nKEOKUK 52632\n(40.39978, -91.387531)",56.0,Lee,...,297,Templeton Rye w/Flask,6,750,$18.09,$27.14,6,$162.84,4.5,1.19
1,S29195400002,11/21/2015,2205,Ding's Honk And Holler,900 E WASHINGTON,CLARINDA,51632,"900 E WASHINGTON\nCLARINDA 51632\n(40.739238, ...",73.0,Page,...,297,Templeton Rye w/Flask,6,750,$18.09,$27.14,12,$325.68,9.0,2.38
2,S29050300001,11/16/2015,3549,Quicker Liquor Store,1414 48TH ST,FORT MADISON,52627,"1414 48TH ST\nFORT MADISON 52627\n(40.624226, ...",56.0,Lee,...,249,Disaronno Amaretto Cavalli Mignon 3-50ml Pack,20,150,$6.40,$9.60,2,$19.20,0.3,0.08


## Política de días sin ventas y `fillna(0)`

En este proyecto diferenciamos el tratamiento de días sin registros según el nivel de agregación:

- **Nivel categoría-estado (todas las tiendas de Iowa)**:
  - Reindexamos la serie a frecuencia diaria.
  - Si un día no tiene ninguna fila de ventas en esa categoría en todo el estado, interpretamos que las ventas fueron **0**.
  - Aplicamos `fillna(0)` sobre la columna de ventas agregadas.

- **Nivel tienda individual**:
  - Un día sin registros en una tienda suele indicar que la tienda estuvo **cerrada** o que no operó.
  - No imputamos automáticamente `0` ventas: esos días se excluyen del entrenamiento de modelos por tienda o se marcan con un flag `es_cierre_tienda = 1`.

Este criterio está documentado como decisión D03 en `DECISIONS.md` y se valida aquí con pequeños ejemplos sintéticos.


In [28]:
# Tests sintéticos de la política de días sin ventas

from datetime import datetime

# Dataset de juguete a nivel de detalle (similar a df_raw)
df_test = pd.DataFrame(
    {
        "Date": ["2024-01-01", "2024-01-03", "2024-01-01" ],
        "Category Name": ["VODKA", "VODKA", "VODKA"],
        "Store Number": [1, 1, 2],
        "Sale (Dollars)": [100.0, 150.0, 200.0],
    }
)

df_test["Date"] = pd.to_datetime(df_test["Date"])

# Serie diaria por categoría (todo Iowa)
cat_daily = (
    df_test
    .groupby("Date", as_index=False)["Sale (Dollars)"].sum()
    .rename(columns={"Sale (Dollars)": "ventas"})
    .set_index("Date")
)

# Reindexamos del 1 al 3 de enero
full_index = pd.date_range("2024-01-01", "2024-01-03", freq="D")
cat_daily_reindexed = cat_daily.reindex(full_index)
cat_daily_filled = cat_daily_reindexed.fillna(0)

# Debug: imprimimos la serie reindexada y completada para diagnósticos
print("cat_daily_filled:")
print(cat_daily_filled)

# Comprobamos explícitamente si la fecha está presente antes de hacer aserciones
if pd.Timestamp("2024-01-02") in cat_daily_filled.index and "ventas" in cat_daily_filled.columns:
    valor_ventas_0201 = cat_daily_filled.loc["2024-01-02", "ventas"]
    assert valor_ventas_0201 == 0.0, f"Esperado 0.0 en 2024-01-02, encontrado {valor_ventas_0201}"
else:
    raise AssertionError("La fila para 2024-01-02 o la columna 'ventas' no existe en cat_daily_filled")

# Serie diaria por tienda
store_daily = (
    df_test
    .groupby(["Store Number", "Date"], as_index=False)["Sale (Dollars)"].sum()
    .rename(columns={"Sale (Dollars)": "ventas"})
)

# Detectamos para cada tienda los días con transacciones
store_daily["Date"] = pd.to_datetime(store_daily["Date"])

# Construimos un calendario completo por tienda
all_dates = full_index
stores = store_daily["Store Number"].unique()

calendar = pd.MultiIndex.from_product(
    [stores, all_dates], names=["Store Number", "Date"]
).to_frame(index=False)

store_daily_full = calendar.merge(
    store_daily,
    on=["Store Number", "Date"],
    how="left",
)

# Definimos día de cierre como día sin registros (ventas NaN)
store_daily_full["es_cierre_tienda"] = store_daily_full["ventas"].isna().astype(int)

# No imputamos 0 donde es cierre; simplemente dejamos NaN para ventas

# Check: para la tienda 1 el 2024-01-02 es cierre, para la tienda 2 el 2024-01-03 es cierre
cierre_t1_0201 = store_daily_full[
    (store_daily_full["Store Number"] == 1) &
    (store_daily_full["Date"] == datetime(2024, 1, 2))
]
cierre_t2_0301 = store_daily_full[
    (store_daily_full["Store Number"] == 2) &
    (store_daily_full["Date"] == datetime(2024, 1, 3))
]

# Debug: imprimimos para diagnóstico
print("\nEstado tienda 1, 2024-01-02:")
print(cierre_t1_0201)
print("\nEstado tienda 2, 2024-01-03:")
print(cierre_t2_0301)

assert int(cierre_t1_0201["es_cierre_tienda"].iloc[0]) == 1
assert int(cierre_t2_0301["es_cierre_tienda"].iloc[0]) == 1

print("Política de días sin ventas validada en ejemplo sintético.")


cat_daily_filled:
            ventas
2024-01-01   300.0
2024-01-02     0.0
2024-01-03   150.0

Estado tienda 1, 2024-01-02:
   Store Number       Date  ventas  es_cierre_tienda
1             1 2024-01-02     NaN                 1

Estado tienda 2, 2024-01-03:
   Store Number       Date  ventas  es_cierre_tienda
5             2 2024-01-03     NaN                 1
Política de días sin ventas validada en ejemplo sintético.


## Control de filas duplicadas por clave compuesta

Antes de construir agregaciones y tablas de trabajo, verificamos la **unicidad** de las filas a nivel de detalle.

La clave compuesta que usamos para detectar duplicados es:

- `Date`
- `Store Number`
- `Category Name`
- `Invoice/Item Number`

Cualquier combinación que aparezca más de una vez se considera potencial duplicado y se analiza/ajusta antes de seguir con el pipeline de forecasting.

In [29]:
# Detección y tratamiento básico de filas duplicadas

clave = ["Date", "Store Number", "Category Name", "Invoice/Item Number"]

# Normalizamos mínimamente los campos relevantes (sin cambiar valores de negocio)
df_raw_dup = df_raw.copy()

# Contamos duplicados estrictos por clave
mask_dup = df_raw_dup.duplicated(subset=clave, keep=False)
num_duplicadas = mask_dup.sum()

print(f"Filas potencialmente duplicadas según clave {clave}: {num_duplicadas:,}")

if num_duplicadas > 0:
    display(df_raw_dup.loc[mask_dup, clave + ["Sale (Dollars)", "Bottles Sold"]].head(10))

# Política mínima actual:
# - Si hay duplicados exactos en la clave, consolidamos sumando métricas numéricas relevantes
#   (por ejemplo, ventas y botellas) para construir una tabla desduplicada.

if num_duplicadas > 0:
    agg_cols = {
        "Sale (Dollars)": "sum",
        "Bottles Sold": "sum",
    }

    otras_cols = [c for c in df_raw_dup.columns if c not in clave + list(agg_cols.keys())]

    # Tomamos la primera ocurrencia para columnas "de referencia" y agregamos numéricas
    df_sin_dup = (
        df_raw_dup
        .groupby(clave, as_index=False)
        .agg({**{c: "first" for c in otras_cols}, **agg_cols})
    )
else:
    df_sin_dup = df_raw_dup.copy()

print(f"Filas tras consolidar duplicados (si los hay): {df_sin_dup.shape[0]:,}")

# Pequeño test sintético: si metemos un duplicado manualmente, el groupby debe reducirlo a 1 fila

df_test_dup = pd.DataFrame(
    {
        "Date": ["2024-01-01", "2024-01-01"],
        "Store Number": [1, 1],
        "Category Name": ["VODKA", "VODKA"],
        "Invoice/Item Number": ["A1", "A1"],
        "Sale (Dollars)": [100.0, 50.0],
        "Bottles Sold": [10, 5],
    }
)

clave_test = ["Date", "Store Number", "Category Name", "Invoice/Item Number"]

df_test_sin_dup = (
    df_test_dup
    .groupby(clave_test, as_index=False)
    .agg({"Sale (Dollars)": "sum", "Bottles Sold": "sum"})
)

assert df_test_sin_dup.shape[0] == 1
assert df_test_sin_dup["Sale (Dollars)"].iloc[0] == 150.0
assert df_test_sin_dup["Bottles Sold"].iloc[0] == 15

print("Lógica de consolidación de duplicados validada en ejemplo sintético.")


Filas potencialmente duplicadas según clave ['Date', 'Store Number', 'Category Name', 'Invoice/Item Number']: 0
Filas tras consolidar duplicados (si los hay): 200,000
Lógica de consolidación de duplicados validada en ejemplo sintético.


## Tablas intermedias en DuckDB y metadatos

A partir del CSV crudo (`Iowa_Liquor_Sales.csv`) construimos tablas intermedias en DuckDB que servirán como base para las agregaciones y features.

Además, registramos **metadatos mínimos** de cada tabla/dataset que generamos (nombre lógico, fecha de generación, versión de código, filtros aplicados y ruta de salida), para poder trazar qué datos alimentan cada ejecución del pipeline.

In [30]:
# Conexión a DuckDB, creación de tabla `raw_ventas` y registro de metadatos

try:
    import duckdb
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install duckdb
    import duckdb

from pathlib import Path
from datetime import datetime
import json

DATA_PATH = "Iowa_Liquor_Sales.csv"  # ya usado en la carga pandas
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

METADATA_PATH = DATA_DIR / "metadata_tablas.json"


def registrar_metadata(tabla: str, path_parquet: str | None = None, filtros: dict | None = None):
    """Registra (append) un entry de metadatos para una tabla/dataset.

    - tabla: nombre lógico de la tabla/dataset
    - path_parquet: ruta relativa al fichero Parquet (si aplica)
    - filtros: diccionario con filtros relevantes (ej. rango de fechas, sample_size)
    """
    record = {
        "tabla": tabla,
        "generated_at": datetime.utcnow().isoformat(timespec="seconds"),
        "code_version": "desarrollo",  # placeholder; se puede reemplazar por hash de git
        "filters": filtros or {},
        "path_parquet": path_parquet,
    }

    try:
        existing = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    except FileNotFoundError:
        existing = []

    existing.append(record)
    METADATA_PATH.write_text(json.dumps(existing, indent=2), encoding="utf-8")
    return record


# Creamos conexión en memoria y tabla base `raw_ventas` (limpia lo mínimo necesario)
con = duckdb.connect(database=":memory:")

limit_clause = f" LIMIT {SAMPLE_SIZE}" if "SAMPLE_SIZE" in globals() and SAMPLE_SIZE is not None else ""

con.execute(f"""
    CREATE OR REPLACE TABLE raw_ventas AS
    WITH csv_cargado AS (
        SELECT 
            "Date",
            "Category Name",
            "Store Number",
            "Invoice/Item Number",
            "Sale (Dollars)",
            "Bottles Sold",
            "State Bottle Cost",
            "State Bottle Retail"
        FROM read_csv_auto('{DATA_PATH}')
        WHERE "Sale (Dollars)" IS NOT NULL AND TRIM("Sale (Dollars)") != ''
        {limit_clause}
    )
    SELECT * FROM csv_cargado
""")

n_raw_duckdb = con.execute("SELECT COUNT(*) AS n FROM raw_ventas").fetchdf()["n"].iloc[0]
print(f"Tabla raw_ventas creada en DuckDB con {n_raw_duckdb:,} filas.")

_ = registrar_metadata(
    tabla="raw_ventas",
    path_parquet=None,
    filtros={"source": DATA_PATH, "sample_size": SAMPLE_SIZE},
)

print(f"Metadatos de 'raw_ventas' registrados en {METADATA_PATH}.")


Tabla raw_ventas creada en DuckDB con 200,000 filas.
Metadatos de 'raw_ventas' registrados en data\metadata_tablas.json.


## Agregaciones diarias y particionado año/mes para ejecuciones incrementales

A partir de `raw_ventas` generamos tablas agregadas de ventas diarias por categoría y por tienda.

- Añadimos columnas `anio` y `mes` derivadas de la fecha.
- Escribimos los resultados en Parquet **particionado por (`anio`, `mes`)**.
- Permitimos ejecuciones incrementales procesando solo fechas posteriores al último día ya presente en los Parquet existentes.

In [31]:
rango_fechas = con.execute("""
    SELECT
        MIN(CAST("Date" AS DATE)) AS min_fecha,
        MAX(CAST("Date" AS DATE)) AS max_fecha
    FROM raw_ventas
""").fetchdf()

rango_fechas

,min_fecha,max_fecha
0,2012-01-03,2015-11-30


In [32]:
# Agregaciones diarias por categoría y por tienda con particionado año/mes

# Rutas de salida
ventas_categoria_dir = DATA_DIR / "ventas_por_categoria_2"
ventas_tienda_dir = DATA_DIR / "ventas_por_tienda_2"

ventas_categoria_dir.mkdir(exist_ok=True, parents=True)
ventas_tienda_dir.mkdir(exist_ok=True, parents=True)

def obtener_ultima_fecha_parquet(base_dir: Path, columna_fecha: str = "fecha"):
    """Devuelve la última fecha presente en los Parquet de base_dir, o None si no hay datos.

    La lectura se hace vía DuckDB para aprovechar el schema.
    """
    if not base_dir.exists():
        return None

    if not any(base_dir.rglob("*.parquet")):
        return None

    query = f"SELECT MAX({columna_fecha}) AS max_fecha FROM read_parquet('{base_dir.as_posix()}/**/*.parquet')"
    try:
        max_fecha = con.execute(query).fetchdf()["max_fecha"].iloc[0]
        return max_fecha
    except Exception:
        return None

# Determinamos última fecha procesada (si ya existen Parquet)
ultima_fecha_cat = obtener_ultima_fecha_parquet(ventas_categoria_dir, columna_fecha="fecha")
ultima_fecha_tienda = obtener_ultima_fecha_parquet(ventas_tienda_dir, columna_fecha="fecha")

print("Última fecha en ventas_por_categoria:", ultima_fecha_cat)
print("Última fecha en ventas_por_tienda:", ultima_fecha_tienda)

# filtro_fecha guarda solo el valor de fecha (YYYY-MM-DD) o None
cutoff_manual = None
filtro_fecha = None

if cutoff_manual is not None:
    filtro_fecha = cutoff_manual
elif ultima_fecha_cat is not None or ultima_fecha_tienda is not None:
    fechas = [f for f in [ultima_fecha_cat, ultima_fecha_tienda] if f is not None]
    if fechas:
        cutoff = min(fechas)
        filtro_fecha = cutoff.date().isoformat()

print("Cutoff incremental:", filtro_fecha)

# Ventas diarias por categoría (todo Iowa) — Fase 2.1: precio, margen, botellas
sql_categoria = """
CREATE OR REPLACE TABLE ventas_categoria AS
WITH agg AS (
    SELECT
        CAST("Date" AS DATE) AS fecha,
        "Category Name" AS categoria,
        SUM(CAST(REPLACE(REPLACE("Sale (Dollars)", '$', ''), ',', '') AS DOUBLE)) AS ventas,
        AVG(TRY_CAST(REPLACE(REPLACE(TRIM(COALESCE("State Bottle Cost", '')), '$', ''), ',', '') AS DOUBLE)) AS precio_costo_medio,
        AVG(TRY_CAST(REPLACE(REPLACE(TRIM(COALESCE("State Bottle Retail", '')), '$', ''), ',', '') AS DOUBLE)) AS precio_retail_medio,
        SUM(TRY_CAST("Bottles Sold" AS INTEGER)) AS botellas_totales,
        EXTRACT(year FROM CAST("Date" AS DATE)) AS anio,
        EXTRACT(month FROM CAST("Date" AS DATE)) AS mes
    FROM raw_ventas
    WHERE (? IS NULL OR CAST("Date" AS DATE) > CAST(? AS DATE))
    GROUP BY fecha, categoria, anio, mes
)
SELECT
    fecha, categoria, ventas, anio, mes,
    precio_costo_medio,
    precio_retail_medio,
    botellas_totales,
    CASE WHEN precio_costo_medio IS NOT NULL AND precio_costo_medio > 0
        THEN (precio_retail_medio - precio_costo_medio) / precio_costo_medio ELSE NULL END AS margen_relativo
FROM agg
"""

# Ventas diarias por tienda (reescrita a usar nombres en el GROUP BY, no índices):
sql_tienda = """
CREATE OR REPLACE TABLE ventas_tienda AS
SELECT
    CAST("Date" AS DATE) AS fecha,
    "Store Number" AS store_id,
    SUM(CAST(REPLACE(REPLACE("Sale (Dollars)", '$', ''), ',', '') AS DOUBLE)) AS ventas,
    EXTRACT(year FROM CAST("Date" AS DATE)) AS anio,
    EXTRACT(month FROM CAST("Date" AS DATE)) AS mes
FROM raw_ventas
WHERE (? IS NULL OR CAST("Date" AS DATE) > CAST(? AS DATE))
GROUP BY fecha, store_id, anio, mes
"""

con.execute(sql_categoria, [filtro_fecha, filtro_fecha])
con.execute(sql_tienda, [filtro_fecha, filtro_fecha])
print("Tablas ventas_categoria y ventas_tienda creadas en DuckDB.")

# Exportamos a Parquet particionado por (anio, mes)
con.execute(f"""
    COPY ventas_categoria TO '{ventas_categoria_dir.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (anio, mes), OVERWRITE_OR_IGNORE TRUE);
""")

con.execute(f"""
    COPY ventas_tienda TO '{ventas_tienda_dir.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (anio, mes), OVERWRITE_OR_IGNORE TRUE);
""")

_ = registrar_metadata(
    tabla="ventas_por_categoria",
    path_parquet=str(ventas_categoria_dir),
    filtros={"incremental_from": str(ultima_fecha_cat) if ultima_fecha_cat is not None else None},
)

_ = registrar_metadata(
    tabla="ventas_por_tienda",
    path_parquet=str(ventas_tienda_dir),
    filtros={"incremental_from": str(ultima_fecha_tienda) if ultima_fecha_tienda is not None else None},
)

print("Parquet particionados por anio/mes generados y metadatos registrados.")


Última fecha en ventas_por_categoria: None
Última fecha en ventas_por_tienda: None
Cutoff incremental: None
Tablas ventas_categoria y ventas_tienda creadas en DuckDB.
Parquet particionados por anio/mes generados y metadatos registrados.


## Cache de features por serie en Parquet

Para evitar recalcular todo el pipeline en cada ejecución, generamos **caches de features por serie**:

- A partir de `ventas_categoria` y `ventas_tienda` construimos tablas con lags, medias móviles y variables de calendario.
- Escribimos estas tablas en Parquet particionado por:
  - Categoría + año/mes (`categoria`, `anio`, `mes`).
  - Tienda + año/mes (`store_id`, `anio`, `mes`).
- Estos caches serán consumidos por `forecasting-licores.ipynb` para entrenar o reentrenar modelos solo en las series/períodos necesarios.

In [33]:
# Generación de features diarios y cache por categoría y por tienda

# Directorios de cache de features
cache_cat_dir = DATA_DIR / "cache_features_categoria"
cache_tienda_dir = DATA_DIR / "cache_features_tienda"

cache_cat_dir.mkdir(exist_ok=True, parents=True)
cache_tienda_dir.mkdir(exist_ok=True, parents=True)

# Creamos vistas con features básicos (lags y medias móviles) en DuckDB

con.execute(
    """
    CREATE OR REPLACE TABLE features_categoria AS
    SELECT
        fecha,
        categoria,
        ventas,
        anio,
        mes,
        EXTRACT(dow FROM fecha) AS dia_semana,
        EXTRACT(month FROM fecha) AS mes_num,
        LAG(ventas, 7) OVER (PARTITION BY categoria ORDER BY fecha) AS lag_7d,
        LAG(ventas, 14) OVER (PARTITION BY categoria ORDER BY fecha) AS lag_14d,
        AVG(ventas) OVER (PARTITION BY categoria ORDER BY fecha ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS roll_mean_7,
        AVG(ventas) OVER (PARTITION BY categoria ORDER BY fecha ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_mean_28
    FROM ventas_categoria
    """
)

con.execute(
    """
    CREATE OR REPLACE TABLE features_tienda AS
    SELECT
        fecha,
        store_id,
        ventas,
        anio,
        mes,
        EXTRACT(dow FROM fecha) AS dia_semana,
        EXTRACT(month FROM fecha) AS mes_num,
        LAG(ventas, 7) OVER (PARTITION BY store_id ORDER BY fecha) AS lag_7d,
        LAG(ventas, 14) OVER (PARTITION BY store_id ORDER BY fecha) AS lag_14d,
        AVG(ventas) OVER (PARTITION BY store_id ORDER BY fecha ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS roll_mean_7,
        AVG(ventas) OVER (PARTITION BY store_id ORDER BY fecha ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_mean_28
    FROM ventas_tienda
    """
)

print("Tablas features_categoria y features_tienda creadas en DuckDB.")

# Exportamos caches particionados por serie + anio/mes

con.execute(f"""
    COPY features_categoria TO '{cache_cat_dir.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (categoria, anio, mes), OVERWRITE_OR_IGNORE TRUE);
""")

con.execute(f"""
    COPY features_tienda TO '{cache_tienda_dir.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (store_id, anio, mes), OVERWRITE_OR_IGNORE TRUE);
""")

_ = registrar_metadata(
    tabla="cache_features_categoria",
    path_parquet=str(cache_cat_dir),
    filtros={},
)

_ = registrar_metadata(
    tabla="cache_features_tienda",
    path_parquet=str(cache_tienda_dir),
    filtros={},
)

print("Caches de features por serie generados en Parquet (categoria/store_id + anio/mes).")


Tablas features_categoria y features_tienda creadas en DuckDB.
Caches de features por serie generados en Parquet (categoria/store_id + anio/mes).


In [34]:
# Vista: ventas semanales POR CATEGORÍA
# date_trunc('week', fecha) agrupa todas las fechas de una misma semana
con.execute("""
    CREATE OR REPLACE VIEW ventas_semanal_categoria AS
    WITH ventas_limpias AS (
        SELECT 
            date_trunc('week', "Date"::DATE)::DATE AS semana,
            "Category Name" AS categoria,
            CAST(REPLACE(REPLACE("Sale (Dollars)", '$', ''), ',', '') AS DOUBLE) AS ventas
        FROM raw_ventas
        WHERE "Sale (Dollars)" IS NOT NULL AND TRIM("Sale (Dollars)") != ''
    )
    SELECT 
        semana,
        categoria,
        SUM(ventas) AS ventas
    FROM ventas_limpias
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

print("Vista ventas_semanal_categoria creada.")
df_cat = con.execute("SELECT * FROM ventas_semanal_categoria LIMIT 10").fetchdf()
df_cat

Vista ventas_semanal_categoria creada.


,semana,categoria,ventas
0,2012-01-02,100 PROOF VODKA,321.21
1,2012-01-02,AMERICAN ALCOHOL,204.51
2,2012-01-02,AMERICAN AMARETTO,180.19
3,2012-01-02,AMERICAN COCKTAILS,2058.31
4,2012-01-02,AMERICAN DRY GINS,1199.54
5,2012-01-02,AMERICAN GRAPE BRANDIES,867.18
6,2012-01-02,AMERICAN SLOE GINS,20.47
7,2012-01-02,APPLE SCHNAPPS,9.45
8,2012-01-02,APRICOT BRANDIES,38.78
9,2012-01-02,BLACKBERRY BRANDIES,201.00


In [35]:
# B1: Metadata geográfica de tiendas
# Genera data/store_metadata.parquet con city, county y n_categorias para features ML
# Este archivo es consumido por forecasting-licores.ipynb para features geográficas (Fase B1)

from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
store_meta_path = DATA_DIR / "store_metadata.parquet"

print("Generando store_metadata.parquet desde tabla raw_ventas...")

try:
    # Reutilizar la conexión DuckDB existente del notebook
    store_meta_df = con.execute("""
        SELECT 
            "Store Number"                              AS store_id,
            ANY_VALUE("Store Name")                     AS store_name,
            UPPER(ANY_VALUE("City"))                    AS city,
            UPPER(ANY_VALUE("County"))                  AS county,
            COUNT(DISTINCT "Category Name")             AS n_categorias_vendidas,
            COUNT(DISTINCT CAST("Date" AS DATE))        AS n_dias_activos,
            SUM(CAST(REPLACE(REPLACE("Sale (Dollars)", '$', ''), ',', '') AS DOUBLE)) AS ventas_total
        FROM raw_ventas
        GROUP BY "Store Number"
        ORDER BY ventas_total DESC
    """).df()
    print(f"  Leído desde raw_ventas (conexión DuckDB existente)")
except Exception as e:
    print(f"  Fallback: creando conexión DuckDB propia ({e})")
    import duckdb
    _con_meta = duckdb.connect()
    store_meta_df = _con_meta.execute("""
        SELECT 
            "Store Number"                              AS store_id,
            ANY_VALUE("Store Name")                     AS store_name,
            UPPER(ANY_VALUE("City"))                    AS city,
            UPPER(ANY_VALUE("County"))                  AS county,
            COUNT(DISTINCT "Category Name")             AS n_categorias_vendidas,
            COUNT(DISTINCT CAST("Date" AS DATE))        AS n_dias_activos
        FROM read_csv_auto('Iowa_Liquor_Sales.csv', ignore_errors=True, sample_size=500000)
        GROUP BY "Store Number"
        ORDER BY n_dias_activos DESC
    """).df()
    _con_meta.close()

# Label encoding para features ML (categorías geográficas → enteros)
store_meta_df["city_enc"]   = store_meta_df["city"].fillna("UNKNOWN").astype("category").cat.codes.astype(int)
store_meta_df["county_enc"] = store_meta_df["county"].fillna("UNKNOWN").astype("category").cat.codes.astype(int)

# store_id como entero (para hacer join en forecasting-licores.ipynb)
store_meta_df["store_id"] = pd.to_numeric(store_meta_df["store_id"], errors="coerce").astype("Int64")

store_meta_df.to_parquet(store_meta_path, index=False)

print(f"\n✓ store_metadata.parquet guardado: {len(store_meta_df)} tiendas → {store_meta_path}")
print(f"  Ciudades únicas:  {store_meta_df['city'].nunique()}")
print(f"  Condados únicos:  {store_meta_df['county'].nunique()}")
print(f"  Columnas: {store_meta_df.columns.tolist()}")
display(store_meta_df.head(10))

Generando store_metadata.parquet desde tabla raw_ventas...
  Fallback: creando conexión DuckDB propia (Binder Error: Referenced column "Store Name" not found in FROM clause!
Candidate bindings: "Store Number", "Category Name", "State Bottle Cost", "State Bottle Retail", "Date"

LINE 4:             ANY_VALUE("Store Name")                     AS store_name,
                              ^)

✓ store_metadata.parquet guardado: 1884 tiendas → data\store_metadata.parquet
  Ciudades únicas:  414
  Condados únicos:  103
  Columnas: ['store_id', 'store_name', 'city', 'county', 'n_categorias_vendidas', 'n_dias_activos', 'city_enc', 'county_enc']


,store_id,store_name,city,county,n_categorias_vendidas,n_dias_activos,city_enc,county_enc
0,2190,"Central City Liquor, Inc.",DES MOINES,POLK,124,1110,100,79
1,4829,Central City 2,DES MOINES,POLK,119,1062,100,79
2,2633,Hy-Vee #3 / BDI / Des Moines,DES MOINES,POLK,124,838,100,79
3,4617,Lickety Liquor,DES MOINES,POLK,101,664,100,79
4,2636,Hy-Vee Wine and Spirits / Hubbell,DES MOINES,POLK,110,649,100,79
5,3952,Lot-A-Spirits,BETTENDORF,SCOTT,116,612,39,85
6,2512,Hy-Vee Wine and Spirits / Iowa City,IOWA CITY,JOHNSON,125,608,188,53
7,3773,Benz Distributing,CEDAR RAPIDS,LINN,124,603,59,58
8,2528,Hy-Vee Food Store #3 / Des Moines,DES MOINES,POLK,116,591,100,79
9,3762,Wine and Spirits Gallery,WINDSOR HEIGHTS,POLK,117,523,407,79
